# CayleyPy Cube555 — native beam on 2×T4

Artgor PieceTransformerQ555 + ResMLPQ, parent-Q blend **0.8 / 0.2**.
The two ranks share one native beam. This is the existing MultiGPUBeamSearch
pipeline with a Cube555 LibTorch scoring module, not the TPU/JAX search engine.
FP16 on T4 can choose different paths from BF16 on TPU. The reported 102/100
results have not been reproduced by this notebook.

Attach `artgor/cube555-tpu-artifacts` and `cayley-py-555-cube`.
The initial beam is conservative; larger widths require memory and quality
validation. All returned solutions are replayed against the original state.
Outputs, both rank logs, checkpoint hashes and submission.csv are retained.


In [ ]:
from pathlib import Path
SOLVER_COMMIT = 'c2c04abac3fab189831f456d2f18b0fbacdfad9a'
SMOKE_TEST = False
PUZZLE_IDS = [1020, 1034]
BEAM_WIDTH = 65536
MAX_DEPTH = 200
TOUCH_BFS_RADIUS = 2
TRANSFORMER_WEIGHT = 0.8


In [ ]:
import json, subprocess, sys, time
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(names) == 2 and all("T4" in name for name in names), names
print("GPUs:", names, "torch:", torch.__version__, flush=True)
repo = Path("/tmp/cube555_native_repo")
if not repo.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git", str(repo)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", SOLVER_COMMIT], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", SOLVER_COMMIT], cwd=repo, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
assert actual == SOLVER_COMMIT, (actual, SOLVER_COMMIT)
sys.path.insert(0, str(repo))
def mount(name, owner=None):
    candidates = [Path("/kaggle/input") / name,
        Path("/kaggle/input/competitions") / name]
    if owner:
        candidates.append(Path("/kaggle/input/datasets") / owner / name)
    found = [p for p in candidates if p.is_dir()]
    if not found:
        raise FileNotFoundError(f"Attach Kaggle input {owner or ''}/{name}")
    return found[0]
assets = mount("cube555-tpu-artifacts", "artgor")
competition = None if SMOKE_TEST else mount("cayley-py-555-cube")
output = Path("/kaggle/working") / ("cube555_" + time.strftime("%Y%m%d_%H%M%S"))
output.mkdir()
(output / "provenance.json").write_text(json.dumps(dict(solver_commit=actual,
    torch=torch.__version__, gpus=names, smoke=SMOKE_TEST), indent=2))


In [ ]:
from tools.cube555.smoke import make_fixture, validate_cuda
if SMOKE_TEST:
    validate_cuda(assets, output / "cuda_parity.json")
    competition = Path("/tmp/cube555_smoke_fixture")
    make_fixture(assets, competition)
    pids, beam, depth, radius = [0, 1, 2, 3], 4096, 6, 0
else:
    pids, beam, depth, radius = PUZZLE_IDS, BEAM_WIDTH, MAX_DEPTH, TOUCH_BFS_RADIUS
command = [sys.executable, "-u", "-m", "tools.cube555.run",
    "--assets", str(assets), "--competition", str(competition),
    "--output", str(output / "run"), "--pids", *map(str, pids),
    "--beam", str(beam), "--depth", str(depth), "--touch-radius", str(radius),
    "--transformer-weight", str(TRANSFORMER_WEIGHT)]
print("Run:", command, flush=True)
with (output / "launcher.log").open("w") as log:
    process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    code = process.wait()
assert code == 0, f"native launcher exited {code}; inspect {output}"


In [ ]:
import pandas as pd
summary = json.loads((output / "run/run_summary.json").read_text())
print(json.dumps(summary, indent=2))
submission = output / "run/submission.csv"
display(pd.read_csv(submission).head(20))
if SMOKE_TEST:
    for pid in range(4):
        solutions = pd.read_csv(output / f"run/puzzle-{pid}/solutions/solutions.csv")
        assert len(solutions) > 0, f"smoke puzzle {pid} not solved"
        assert solutions["valid"].all(), f"smoke puzzle {pid} failed replay"
    print("PASS: four legal Cube555 scrambles solved and replayed by two native ranks")
